# AutoML-IDS — CICIDS2017 Dataset
Full autonomous intrusion detection pipeline on the CICIDS2017 benchmark.

**Pipeline steps:**
1. Data loading & inspection
2. Preprocessing (label encoding, missing values)
3. Initial base-model training (for feature importance)
4. Automated feature selection (90 % cumulative importance)
5. Data balancing with TVAE
6. Re-training on balanced, feature-selected data
7. Hyperparameter optimisation (BO-TPE)
8. Automated model selection & ensemble (OCSE)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os

# Resolve project root whether Jupyter was launched from project root or notebooks/
_cwd = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src import (
    DataLoader, DataPreprocessor, FeatureSelector,
    DataBalancer, ModelTrainer, HyperparameterTuner,
    ModelSelector, EnsembleBuilder, ModelEvaluator,
)

## 1  Load data

In [ ]:
loader = DataLoader('data/raw/CICIDS2017_sample_0.02.csv', label_col='Label')
df = loader.load()
loader.summary()

## 2  Preprocess

In [ ]:
prep = DataPreprocessor(label_col='Label')
df   = prep.encode_labels(df)
df   = prep.handle_missing(df)

feature_names = df.drop(['Label'], axis=1).columns.tolist()
X = df[feature_names].values
y = np.ravel(df['Label'].values)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 3  Initial model training (feature importance)

In [ ]:
trainer_fs = ModelTrainer(cv=3)
trainer_fs.train_all(X_train, y_train, X_test)

## 4  Automated feature selection

In [ ]:
selector_fs = ModelSelector(top_k=3)
selector_fs.fit(trainer_fs.cv_scores)
top_models_fs = selector_fs.top_models
print('Top-3 models for feature selection:', top_models_fs)


In [ ]:
fs = FeatureSelector(importance_threshold=0.9)
fs.fit(top_models_fs, trainer_fs.trained_models, feature_names)
print(f'Selected {len(fs.selected_features)} features:', fs.selected_features)


In [ ]:
fs.plot(
    title='Feature Importance — CICIDS2017',
    save_path='output/plots/cicids2017_feature_importance.png',
)

In [ ]:
# Rebuild train/test with selected features and save for TVAE metadata
X_fs = df[fs.selected_features].values
X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(
    X_fs, y, test_size=0.2, random_state=0, stratify=y
)

fs_csv = 'data/processed/cicids2017_fs.csv'
df_fs  = pd.DataFrame(X_fs, columns=fs.selected_features)
df_fs['Label'] = y
df_fs.to_csv(fs_csv, index=False)
print('Feature-selected dataset saved to', fs_csv)

## 5  Data balancing with TVAE

In [ ]:
X_train_df = pd.DataFrame(X_train_fs, columns=fs.selected_features)
y_train_s  = pd.Series(y_train_fs)

print('Class distribution before balancing:')
print(y_train_s.value_counts())


In [ ]:
balancer = DataBalancer(label_col='Label')
X_train_bal, y_train_bal = balancer.fit_resample(
    X_train_df, y_train_s, fs_csv
)

print('Class distribution after balancing:')
print(pd.Series(y_train_bal).value_counts())

## 6  Model training on balanced data

In [ ]:
trainer = ModelTrainer(cv=3)
trainer.train_all(X_train_bal.values, y_train_bal.values, X_test_fs)

## 7  Hyperparameter optimisation (BO-TPE)

In [ ]:
tuner = HyperparameterTuner(max_evals=20)

for name in ModelTrainer.MODEL_KEYS:
    tuned_model, best = tuner.tune(
        name,
        X_train_bal.values, y_train_bal.values,
        X_test_fs, y_test_fs,
    )
    trainer.replace_model(name, tuned_model, X_train_bal.values, X_test_fs)

## 8  Evaluation — base models

In [ ]:
evaluator = ModelEvaluator(output_dir='output')
results   = {}

for name, model in trainer.trained_models.items():
    res = evaluator.evaluate(model, X_test_fs, y_test_fs, name)
    evaluator.plot_confusion_matrix(y_test_fs, res['predictions'], name)
    results[name] = res

## 9  Automated model selection & ensemble (OCSE)

In [ ]:
selector_ens = ModelSelector(top_k=3)
selector_ens.fit(trainer.cv_scores)
top_models_ens = selector_ens.top_models
print('Top-3 models for ensemble:', top_models_ens)

In [ ]:
builder = EnsembleBuilder()

stk1, pred1 = builder.traditional_stacking(
    trainer.predictions, top_models_ens, y_train_bal.values, y_test_fs
)
stk2, pred2 = builder.confidence_stacking(
    trainer.predictions, top_models_ens, y_train_bal.values, y_test_fs
)
stk3, pred3 = builder.hybrid_stacking(
    trainer.predictions, top_models_ens, y_train_bal.values, y_test_fs
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

for label, pred in [
    ('Traditional Stacking',   pred1),
    ('Confidence Stacking',    pred2),
    ('Hybrid Stacking (OCSE)', pred3),
]:
    acc              = accuracy_score(y_test_fs, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test_fs, pred, average='weighted')
    results[label]   = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

evaluator.compare_models(results)